# L4a: Graph and Tree Representations

A graph is a set of objects, called vertices, and a set of pairwise connections between them, called edges. Roads between cities, reactions between chemical species, and calls between functions are all graphs. A tree is the simplest connected graph: it has just enough edges to reach every vertex and no cycle. Today we define the vocabulary, look at the common graph families, and see how a graph is stored.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Describe a graph and its basic measures:__ Define vertices, edges, paths, and connectivity, and tell a directed graph from an undirected one. Compute the degree of a vertex and the density of a graph from the vertex and edge counts.
> * __Recognize complete graphs, bipartite graphs, and trees:__ State what makes each family special: every pair joined, edges only between two groups, or connected with no cycle. Explain what the edge count and the coloring or path properties of each family tell an algorithm.
> * __Choose a storage representation:__ Read a graph from an edge list into an adjacency list and an adjacency matrix. Choose between the two from the density of the graph and from whether the algorithm mostly checks for a single edge or mostly loops over the neighbors of a vertex.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
# Setup -
include(joinpath(@__DIR__, "Include.jl")); # activate the pinned course environment and load the L4a graph helpers

This lecture needs nothing beyond the course environment: [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) supplies the checks we run on the storage example, and the course package supplies the graph functions we call, which live in [the `GraphRepresentation.jl` file](../../../code/src/GraphRepresentation.jl). The edge list we read is in the `data` folder of this meeting.
___

## Simple Graphs
A simple graph $\mathcal{G} = (\mathcal{V},\mathcal{E})$ has a set of vertices $\mathcal{V}$ and a set of edges $\mathcal{E}$. Each edge joins two different vertices, and no edge is repeated, so there are no self-loops and no parallel edges. In a directed graph, one edge each way between two vertices counts as two different edges. An edge can carry a weight, such as a distance or a cost, or carry no weight at all.

Edges can also have a direction:
* In a __directed__ graph, each edge points from one vertex to another. In a social network, a directed edge can mean that person A follows person B, while B need not follow A.
* In an __undirected__ graph, an edge has no direction, and the connection reads the same from both ends.

A __path__ from $v_{i}$ to $v_{j}$ is a sequence of distinct vertices that starts at $v_{i}$ and ends at $v_{j}$, where each consecutive pair is joined by an edge. In a directed graph, each edge on the path must be followed in its direction. A graph is __connected__ if there is a path between every pair of vertices; for a directed graph we ignore the edge directions when asking this.

<div>
    <center>
        <img src="figs/Fig-General-Graph-Schematic.svg" width="980" alt="Left: an undirected graph with six vertices and seven weighted edges, with the degree of vertex 4 marked. Right: the same vertices and edges drawn as a directed acyclic graph, with the in-degree of vertex 2 and the out-degree of vertex 4 marked."/>
    </center>
</div>

The figure shows the same six vertices and seven edges drawn twice, once undirected and once directed, and marks the degree counts we define next.

### Graph Properties and Metrics
A few counts describe how connected a graph is, both at a single vertex and overall. The __degree__ of a vertex $v_{i}\in\mathcal{V}$, written $\deg(v_{i})$, is the number of edges that touch it. In a directed graph we count the two directions separately:
* __In-degree__ $\deg^{\text{in}}(v_{i})$: the number of edges that point into $v_{i}$.
* __Out-degree__ $\deg^{\text{out}}(v_{i})$: the number of edges that point out of $v_{i}$.

The total degree of a vertex in a directed graph is $\deg(v_{i}) = \deg^{\text{in}}(v_{i}) + \deg^{\text{out}}(v_{i})$. In the figure above, vertex 4 has degree 2 in the undirected drawing, and in the directed drawing vertex 2 has in-degree 2 and vertex 4 has out-degree 1.

> __Core graph measures:__
>
> Each edge has two ends, so the total degrees obey the __handshaking lemma__. For a directed graph, the in-degrees and out-degrees each count every edge once:
> $$
> \begin{align*}
> \sum_{v_i\in\mathcal{V}}\deg(v_i) &= 2|\mathcal{E}|,\\
> \sum_{v_i\in\mathcal{V}}\deg^{\mathrm{in}}(v_i) &= \sum_{v_i\in\mathcal{V}}\deg^{\mathrm{out}}(v_i)=|\mathcal{E}|.
> \end{align*}
> $$
>
> The __minimum degree__ $\delta(\mathcal{G})$ and __maximum degree__ $\Delta(\mathcal{G})$ bound the __average degree__ $\bar d(\mathcal{G})$:
> $$
> \begin{align*}
> \bar d(\mathcal{G}) &= \frac{1}{|\mathcal{V}|}\sum_{v_i\in\mathcal{V}}\deg(v_i)=\frac{2|\mathcal{E}|}{|\mathcal{V}|},\\
> \delta(\mathcal{G}) &\leq \bar d(\mathcal{G}) \leq \Delta(\mathcal{G}).
> \end{align*}
> $$
> A graph is __regular__ of degree $r$ when every vertex has degree $r$. With $n=|\mathcal{V}|$ vertices, the maximum edge count and the __density__ $\rho(\mathcal{G})$ are
> $$
> \begin{align*}
> |\mathcal{E}|_{\max} &= \begin{cases}n(n-1)/2 & \text{undirected},\\ n(n-1) & \text{directed},\end{cases}\\
> \rho(\mathcal{G}) &= \frac{|\mathcal{E}|}{|\mathcal{E}|_{\max}}.
> \end{align*}
> $$
> A __dense__ graph has $\rho$ close to 1, whereas a __sparse__ graph has $\rho$ close to 0. When $n<2$, there are no possible edges, so we take $\rho(\mathcal{G})=0$, which is also what the course code returns.

Four additional parameters describe the shape of a graph rather than only its degrees. The diameter requires a connected graph; the other three are defined for any undirected graph.

* __Diameter__ $\text{diam}(\mathcal{G})$: the largest distance between any two vertices, where the distance is the number of edges on a shortest path. It is the worst case for getting from one vertex to another.
* __Clique number__ $\omega(\mathcal{G})$: the size of the largest set of vertices in which every pair is joined, such as the largest group of people who all know each other.
* __Chromatic number__ $\chi(\mathcal{G})$: the fewest colors needed so that no edge joins two vertices of the same color, such as the fewest time slots that schedule a set of classes with no student in two classes at once.
* __Independence number__ $\alpha(\mathcal{G})$: the size of the largest set of vertices with no edge among them, such as the most activities that can run at the same time with no conflict.

We will meet each of these again in the graph families that follow. First, let's look at how a graph is stored.
___

## How are Graphs Stored?
An algorithm needs the graph in memory in a form it can query. Three representations are common: the edge list, the adjacency matrix, and the adjacency list.

An __edge list__ is the simplest form: one record per edge, holding the source vertex, the target vertex, and the weight if there is one. It is how a graph is usually written to a file, and it is what we read in the example at the end of this section. It is a poor form to compute on, because finding the neighbors of a vertex means scanning every record.

An __adjacency matrix__ $\mathbf{A}$ for a graph with $|\mathcal{V}|$ vertices is a $|\mathcal{V}|\times|\mathcal{V}|$ matrix. The entry $a_{ij}$ in row $i$ and column $j$ describes the edge from $v_{i}$ to $v_{j}$:
* __Unweighted__: $a_{ij}=1$ if the edge exists and $a_{ij}=0$ if it does not.
* __Weighted__: $a_{ij}=w_{ij}$, the weight of the edge, if the edge exists, and $a_{ij}=0$ if it does not.

For an undirected graph the matrix is symmetric, $a_{ij} = a_{ji}$; for a directed graph it need not be. The weighted form has one blind spot: a stored zero could mean either a missing edge or an edge of weight zero, so a graph with zero-weight edges needs a separate marker for missing edges.

An __adjacency list__ is a dictionary with one entry per vertex. The entry for $v_{i}$ holds the set $\mathcal{C}_{i}$ of vertices that $v_{i}$ is joined to. In a directed graph these are the vertices its edges point to, the out-neighbors. When weights are needed, they are stored beside each neighbor.

> __What does it cost to store and query a graph?__
>
> The object being stored is the connectivity of the same graph $\mathcal{G}=(\mathcal{V},\mathcal{E})$: its $n=|\mathcal{V}|$ vertex identifiers and $m=|\mathcal{E}|$ edges, together with an edge weight when the graph is weighted. The representation changes how those connections are laid out in memory.
>
> * An __edge list__ stores one `(source, target, weight)` record for each edge, so it uses $O(m)$ records. It is compact and natural for a file, but either finding one particular edge or collecting every edge leaving $v_i$ requires scanning as many as $m$ records.
> * An __adjacency matrix__ stores one entry $a_{ij}$ for every ordered pair of vertices, so it uses exactly $n^2$ entries whether or not most pairs are connected. The entry for $(v_i,v_j)$ can be read in $O(1)$ time, while finding every neighbor of $v_i$ requires scanning its $n$ matrix entries.
> * An __adjacency list__ stores one neighbor collection for each vertex and one target entry for each directed edge, so it uses $n+m$ stored items. The `Dict{Int64,Vector{Int64}}` used in this lecture returns the out-neighbors of $v_i$ in $O(\deg^{\mathrm{out}}(v_i))$ time; testing for one target also requires scanning that vector. A weighted list would store a `(target, weight)` pair instead of only the target.
>
> Ignoring container overhead and the constant number of fields in each edge record, the storage requirements are therefore
> $$
> S_{\mathrm{edge\ list}}=O(m),\qquad S_{\mathrm{matrix}}=O(n^2),\qquad S_{\mathrm{adjacency\ list}}=O(n+m).
> $$
> For an undirected adjacency list, each edge appears in both endpoint lists, giving $n+2m$ stored items; the asymptotic requirement remains $O(n+m)$.

The representation should therefore follow both the graph and the computation. A matrix suits a dense graph or an algorithm that repeatedly asks whether a particular edge exists. A list suits a sparse graph or an algorithm that repeatedly asks for all neighbors of the current vertex. Breadth-first and depth-first search make the second query at every visited vertex, so the L4b traversal algorithms use an adjacency list.

### Worked Example: One Graph in Three Forms
The file `SimpleGraph.txt` in the `data` folder of this meeting is an edge list for a small directed graph with six vertices and seven weighted edges. The next lab uses a copy of the same file. We read it with [the `read_weighted_edges(...)` function](../../../code/src/GraphRepresentation.jl), then build the two computational forms with [the `adjacency_list(...)` function](../../../code/src/GraphRepresentation.jl) and [the `adjacency_matrix(...)` function](../../../code/src/GraphRepresentation.jl).

The adjacency list keeps only the out-neighbors of each vertex and drops the weights; the matrix keeps the weights. Both functions take the vertex set from the edge endpoints, so a vertex with no edges would not appear. The cell stores the edge records in `edge_records::Vector{<:NamedTuple}`, one `(source, target, weight)` record per edge, the adjacency list in `adjacency::Dict{Int64, Vector{Int64}}`, and the matrix with its vertex order in `matrix_representation::NamedTuple`.

In [ ]:
# Build three representations of the same directed, weighted graph -
# The let block keeps temporary names local; its final tuple is unpacked into the three notebook variables.
edge_records, adjacency, matrix_representation = let
    # Locate and read the edge list -
    edge_path = joinpath(CHEME5800_L4A_DATA, "SimpleGraph.txt") # path anchored to the L4a data directory, not pwd()
    records = read_weighted_edges(edge_path)                    # Vector of (source, target, weight) NamedTuples

    # Convert the edge records into two computational representations -
    list = adjacency_list(records)                               # Dict: vertex id => sorted outgoing-neighbor ids
    matrix = adjacency_matrix(records)                           # NamedTuple: weighted matrix plus row/column vertex ids

    # Return values from the local scope -
    records, list, matrix                                       # return the three representations from let
end; # suppress the full construction output; display the useful fields in the next cell

Let's look at the two forms side by side. The `vertex_order` entry says which vertex each row and column of the matrix belongs to.

In [ ]:
# Display the out-neighbor list beside the equivalent weighted matrix -
(
    adjacency = adjacency,                            # each key is a vertex; each value is its sorted out-neighbor vector
    vertex_order = matrix_representation.vertex_ids, # position i in this vector identifies matrix row and column i
    matrix = matrix_representation.matrix,            # entry (i,j) is the weight from vertex_order[i] to vertex_order[j]
)

Do we see what we expect? Vertex 1 points to vertices 2 and 3, and row 1 of the matrix holds the weights 10 and 100 in columns 2 and 3. Vertex 6 has an empty list because no edge leaves it.

Now let's count what each form stores. [The `representation_report(...)` function](../../../code/src/GraphRepresentation.jl) returns the vertex and edge counts, the directed density $|\mathcal{E}|/(|\mathcal{V}|(|\mathcal{V}|-1))$, the number of matrix entries $|\mathcal{V}|^{2}$, and the number of adjacency-list entries $|\mathcal{V}| + |\mathcal{E}|$, counting one dictionary key per vertex and one neighbor slot per edge. The same cell also sizes both forms for 100,000 vertices with ten edges each, at 8 bytes per entry and ignoring container overhead. The cell stores the report in `representation::NamedTuple`.

In [ ]:
# Compare storage for the observed graph and a larger sparse graph -
# The let block exposes only the six-vertex report; the large-graph variables are temporary teaching calculations.
representation = let
    # Measure the graph read from SimpleGraph.txt -
    report = representation_report(edge_records)                    # graph measures and storage-entry counts

    # Specify a larger sparse graph for an order-of-magnitude comparison -
    n, k = 100_000, 10                                              # n vertices, k outgoing edges each, and m = n⋅k
    bytes_per_entry = sizeof(Float64)                               # Float64 weights and Int64 ids each use 8 bytes

    # Estimate storage while deliberately ignoring container overhead -
    matrix_gigabytes = n^2 * bytes_per_entry / 1e9                  # n² weighted entries, converted to decimal gigabytes
    adjacency_list_megabytes = (n + n * k) * bytes_per_entry / 1e6  # n keys plus n⋅k targets, in decimal MB

    # Display the large-graph comparison and return the measured small-graph report -
    println("Large sparse graph: matrix ≈ $(matrix_gigabytes) GB, adjacency list ≈ $(adjacency_list_megabytes) MB")
    report                                                         # return the small-graph NamedTuple from let
end

So what do we see? For six vertices the matrix holds 36 entries against 13 for the list, a small gap. For the large sparse graph the matrix would need about 80 gigabytes against about 9 megabytes for the list, which is the whole argument for adjacency lists on sparse graphs. The checks confirm the counts, the density, and two entries we can read off the edge list by hand.

In [ ]:
# Check that every representation encodes the expected graph -
@testset "graph representations" begin
    # Check the structure encoded by the input and both representations -
    @test representation.vertices == 6                       # endpoint records contain the six vertex ids 1,...,6
    @test representation.edges == 7                          # the file contains seven directed source-to-target records
    @test representation.density ≈ 7 / 30                    # 7 observed edges divided by 6(6-1) possible loop-free edges
    @test adjacency[1] == [2, 3]                             # vertex 1 points outward to vertices 2 and 3, in sorted order
    @test matrix_representation.matrix[1, 2] == 10.0         # entry (1,2) stores the weight of edge 1 → 2

    # Check the exact storage counts used in the lecture comparison -
    @test representation.matrix_entries == 36                # a 6×6 matrix reserves one entry for every ordered pair
    @test representation.adjacency_list_entries == 13        # six dictionary keys plus one target entry per observed edge
end

___

## Graph Families
The graph measures above become especially informative when the edge set has a recognizable structure. Complete graphs, bipartite graphs, and trees are three useful reference families: a complete graph contains every possible edge, a bipartite graph permits edges only across two vertex groups, and a tree contains exactly the edges needed for connectivity.

__Complete graphs.__ A complete graph $K_{n}$ is a simple undirected graph on $n$ vertices in which every pair of distinct vertices is joined. Once $n$ is known, the entire structure is fixed: there is only one complete graph of that size up to the names assigned to its vertices.

> __Complete graphs are the dense extreme:__
>
> Every vertex touches the other $n-1$ vertices, so $K_n$ is regular of degree $n-1$. Every possible edge is present, every pair of vertices is one step apart, the full vertex set is a clique, and a proper coloring requires a different color for every vertex. For $n\geq 2$, these facts give
> $$
> \begin{align*}
> |\mathcal{E}| &= \binom{n}{2}=\frac{n(n-1)}{2}, & \deg(v_i) &= n-1,\\
> \rho(K_n) &= 1, & \operatorname{diam}(K_n) &= 1.
> \end{align*}
> $$
> The clique, chromatic, and independence numbers are therefore
> $$
> \omega(K_n)=\chi(K_n)=n,\qquad \alpha(K_n)=1.
> $$

This all-to-all structure appears directly in a round-robin tournament, where every team plays every other team. It also supplies a worst-case input for algorithms whose work grows with the number of edges. In optimization, the traveling-salesman problem is commonly posed on a weighted complete graph: every city can be reached directly, and the problem is to choose the least costly tour.

Complete graphs place no restriction on which distinct vertices may be adjacent. Bipartite graphs introduce the opposite kind of organization by allowing an edge only when its endpoints belong to different groups.

__Bipartite graphs.__ A graph $\mathcal{G}=(\mathcal{V},\mathcal{E})$ is bipartite if its vertex set can be partitioned into two disjoint parts, $\mathcal{V}_{1}$ and $\mathcal{V}_{2}$, so that every edge has one endpoint in each part. No edge joins two vertices within $\mathcal{V}_{1}$ or two vertices within $\mathcal{V}_{2}$.

<div>
    <center>
        <img src="figs/Fig-Bipartite-Graph-Schematic.png" width="280" alt="A bipartite graph drawn with four vertices in a left column and eight in a right column; every edge crosses from the left column to the right column."/>
    </center>
</div>

The figure makes the partition visible by placing the four vertices of $\mathcal{V}_{1}$ on the left and the eight vertices of $\mathcal{V}_{2}$ on the right. Every edge crosses between the two columns. A __complete bipartite graph__ $K_{m,n}$ contains every one of the possible cross-part edges when $|\mathcal{V}_{1}|=m$ and $|\mathcal{V}_{2}|=n$.

A __matching__ selects edges that share no vertices, so it describes a set of pairings in which no object is used twice. In an assignment graph, a matching that covers $\mathcal{V}_{1}$ assigns every worker or student on the left to a distinct option on the right.

> __Bipartite structure links coloring, cycles, and matching:__
>
> For an undirected graph $\mathcal{G}$, the following are equivalent:
> 1. $\mathcal{G}$ is bipartite.
> 2. $\mathcal{G}$ can be properly colored with two colors, so $\chi(\mathcal{G}) \leq 2$.
> 3. $\mathcal{G}$ has no cycle of odd length.
>
> Coloring the two parts with different colors proves $1\Rightarrow2$, while an odd cycle makes alternating two colors impossible. This equivalence turns a structural definition into an algorithmic test.
>
> In $K_{m,n}$, each vertex of $\mathcal{V}_{1}$ is adjacent to all $n$ vertices of $\mathcal{V}_{2}$, and each vertex of $\mathcal{V}_{2}$ is adjacent to all $m$ vertices of $\mathcal{V}_{1}$. Thus
> $$
> |\mathcal{E}(K_{m,n})|=mn,\qquad \deg(v)=\begin{cases}n & v\in\mathcal{V}_{1},\\ m & v\in\mathcal{V}_{2}.\end{cases}
> $$
> The graph $K_{m,n}$ is regular exactly when $m=n$.
>
> __Hall's theorem__ characterizes when a matching covers every vertex of $\mathcal{V}_{1}$. If $N(S)\subseteq\mathcal{V}_{2}$ denotes all neighbors of a subset $S\subseteq\mathcal{V}_{1}$, such a matching exists if and only if
> $$
> |N(S)|\geq |S|\qquad\text{for every }S\subseteq\mathcal{V}_{1}.
> $$
> In words, every group on the left must collectively have at least as many options on the right as the number of members in the group. A matching is __perfect__ when it covers both parts, which also requires $|\mathcal{V}_{1}|=|\mathcal{V}_{2}|$.
>
> The partition also appears in the adjacency matrix. After listing the vertices of $\mathcal{V}_{1}$ before those of $\mathcal{V}_{2}$, an undirected bipartite graph has the block form
> $$
> \mathbf{A}=\begin{bmatrix}\mathbf{0} & \mathbf{B}\\ \mathbf{B}^{\top} & \mathbf{0}\end{bmatrix},
> $$
> where the $m\times n$ __biadjacency matrix__ $\mathbf{B}$ contains all nonzero information. The two diagonal blocks are zero because no edge stays within either part.

The coloring equivalence gives a direct traversal-based test for bipartiteness:
1. Mark every vertex uncolored.
2. Choose an uncolored vertex, assign it color 1, and traverse outward from it with breadth-first or depth-first search.
3. Give each uncolored neighbor the opposite color of the current vertex. If an edge joins two vertices with the same color, stop: the graph is not bipartite.
4. Repeat from another uncolored vertex until every connected component has been tested. If no conflict occurs, the two colors recover the two parts.

With an adjacency list, this test visits each vertex once and examines each edge at most twice, so its running time is $O(|\mathcal{V}|+|\mathcal{E}|)$. This is the first direct use of the traversal algorithms developed in L4b.

The same two-part interpretation supports several applications without changing the underlying mathematics. Assignment graphs connect workers to jobs or students to courses; recommendation graphs connect users to items; and biological graphs can connect genes to the proteins they regulate or species to the habitats they occupy. An adjacency list needs no special bipartite format, but the matrix representation can exploit $\mathbf{B}$ instead of storing the two zero blocks and the duplicate transpose block.

__Trees.__ Complete graphs use every possible edge; trees sit at the sparse connected extreme. A tree $\mathcal{T}=(\mathcal{V},\mathcal{E})$ is a connected undirected graph with no cycle. Connectivity prevents isolated pieces, while the absence of cycles prevents redundant routes. Consequently, exactly one path joins each pair of vertices. Removing any edge breaks that path and disconnects the tree; adding any new edge closes exactly one cycle around the path that already connected its endpoints.

> __Equivalent ways to recognize a tree:__
>
> For an undirected graph $\mathcal{G}$ with $n$ vertices, each statement below implies all the others:
> 1. $\mathcal{G}$ is a tree.
> 2. $\mathcal{G}$ is connected and has exactly $n-1$ edges.
> 3. $\mathcal{G}$ has no cycle and has exactly $n-1$ edges.
> 4. $\mathcal{G}$ is connected, and removing any edge disconnects it.
> 5. $\mathcal{G}$ has no cycle, and adding any edge creates exactly one cycle.
> 6. Any two vertices of $\mathcal{G}$ are joined by exactly one path.
>
> These characterizations all force the same edge count. For $n\geq2$, they also determine the density:
> $$
> |\mathcal{E}|=n-1,\qquad \rho(\mathcal{G})=\frac{n-1}{n(n-1)/2}=\frac{2}{n}.
> $$
> Thus a tree remains connected while its density decreases toward zero as the number of vertices increases.

Choosing one vertex as the __root__ gives the undirected tree a hierarchy without changing its edges. Every other vertex has one __parent__, the next vertex on its unique path to the root, and zero or more __children__. A vertex with no children is a __leaf__, and the __height__ of the rooted tree is the number of edges on its longest root-to-leaf path.

<div>
    <center>
        <img src="figs/Fig-General-Tree-Schematic.svg" width="880" alt="A rooted tree drawn top down: the root at height 0, branch nodes at heights 1 and 2, and leaves at heights 2 and 3, with the empty set marking the missing children of each leaf."/>
    </center>
</div>

The figure shows a rooted tree of height 3. The unique-parent structure is why rooted trees model file systems, organization charts, and the calls generated by a recursive function.

Trees also expose useful structure inside general graphs. Every connected graph $\mathcal{G}$ contains at least one __spanning tree__: a subgraph $\mathcal{T}$ that retains every vertex of $\mathcal{G}$ but only enough edges to stay connected. If an acyclic graph is not connected, each connected component is a tree and their disjoint union is called a __forest__.

This constrained structure often makes algorithms simpler. For example, a largest independent set can be found in polynomial time on a tree even though the corresponding problem on a general graph has no known polynomial-time algorithm. The next lab begins with a more basic operation shared by trees and general graphs: systematically visiting every vertex reachable from a chosen start.
___

## Lab Exercises
The definitions in this lecture describe the static structure of a graph; traversal turns that structure into a computation. Starting from one vertex, a traversal repeatedly asks for the current vertex's neighbors and continues until it has visited every reachable vertex. This repeated neighbor query is precisely the operation for which an adjacency list is efficient.

In L4b we use the same six-vertex directed graph and implement the two standard traversal orders. Breadth-first search (BFS) keeps vertices waiting in a queue and explores the graph layer by layer; depth-first search (DFS) follows one branch before backtracking, using a stack or the program's recursion stack. Both algorithms run in $O(|\mathcal{V}|+|\mathcal{E}|)$ time when the graph is stored as an adjacency list.
___

## Summary
A graph is a set of vertices and a set of edges between them, and the way it is stored decides which questions an algorithm can answer quickly.

> __Key Takeaways:__
>
> * __Describe a graph by counting:__ The degree of a vertex counts its edges, and the density of a graph is the fraction of possible edges that are present. The handshaking lemma ties the two together, since the degrees add up to twice the edge count.
> * __Three families set the extremes:__ A complete graph has every possible edge, and a tree has one fewer edge than it has vertices, the fewest that keep it connected. An undirected graph is bipartite, with edges only between two groups, exactly when it has no odd cycle.
> * __Choose storage from density and the operations:__ An adjacency matrix answers whether one edge exists in constant time and suits dense graphs. An adjacency list takes space proportional to the vertex and edge counts, lists the neighbors of a vertex directly, and suits the sparse graphs that traversal algorithms walk over.

Next, in lab, we use the adjacency list to traverse a graph with breadth-first and depth-first search.
___